### Import Tokens and Dataset

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
VLLM_API_KEY=os.getenv("VLLM_API_KEY")
HF_TOKEN=os.getenv("HF_TOKEN")

In [ ]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("vblagoje/PubMedQA_instruction", split="train")
dataset = dataset.select(range(1000))
all_documents = [Document(content=doc["context"]) for doc in dataset]
all_questions = [doc["instruction"] for doc in dataset]
all_ground_truth_answers = [doc["response"] for doc in dataset]

Set the models to use:

In [ ]:
if VLLM_API_KEY is not None:
    API_URL = "https://orfeo-llm.areasciencepark.it/vllm/v1"
    EMBED_MODEL_ID = "Qwen/Qwen3-VL-Embedding-8B"
    GENERATION_MODEL_ID = "Qwen/Qwen3.5-9B"
    JUDGE_MODEL_ID = "google/gemma-4-26B-A4B-it"
else:
    # EMBED_MODEL_ID = "google/embeddinggemma-300m" # DESKTOP
    EMBED_MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-8bit" # MAC
    # GENERATION_MODEL_ID = "cyankiwi/Qwen3.5-4B-AWQ-4bit" # DESKTOP
    GENERATION_MODEL_ID = "Qwen/Qwen3-0.6B" # MAC/DESKTOP

# Indexing data into Qdrant

In [ ]:
from haystack import Pipeline
from haystack.utils import Secret
from haystack.components.writers import DocumentWriter

In [ ]:
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

Remember to change the parameters based on the actual model used and on the saving method.

In [ ]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(None, # ":memory" if stored on memory, None is stored on disk
                                     embedding_dim=4096, # 768 for gemma, 1024 credo for Qwen, 4096 for Qwen3-8B
                                     path="./.qdrant/",
                                     on_disk=True)

In [ ]:
converter = DoclingConverter(export_type=ExportType.DOC_CHUNKS)

indexing_embedder = VLLMDocumentEmbedder(model=EMBED_MODEL_ID,
                                         api_key=Secret.from_token(VLLM_API_KEY),
                                         api_base_url=API_URL)
                                        #  api_base_url="http://localhost:8001/v1")

writer = DocumentWriter(document_store=document_store)

> The `DoclingConverter` is not used at the moment because data is already prepared for embedding

In [ ]:
# indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("embedder", indexing_embedder)
# indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

In [ ]:
indexing_pipeline.run(
    {
        "embedder": {"documents": all_documents},
    })

# RAG

In [ ]:
from haystack.components.builders import AnswerBuilder, ChatPromptBuilder
from haystack.dataclasses import ChatMessage

In [ ]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [ ]:
template = [
    ChatMessage.from_user(
        """
        You have to answer the following question based on the given context information only.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}

        Question: {{question}}
        Answer:
        """
    )
]

rag_pipeline = Pipeline()

In [ ]:
rag_embedder = VLLMTextEmbedder(model=EMBED_MODEL_ID,
                                api_key=Secret.from_token(VLLM_API_KEY),
                                api_base_url=API_URL)
                                # api_base_url="http://localhost:8001/v1")

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=3)

prompt_builder = ChatPromptBuilder(template=template, required_variables={"question", "documents"})

llm = VLLMChatGenerator(model=GENERATION_MODEL_ID,
                        api_key=Secret.from_token(VLLM_API_KEY),
                        api_base_url=API_URL)
                        # api_base_url="http://localhost:8000/v1")

answer_builder = AnswerBuilder()

In [ ]:
rag_pipeline.add_component("query_embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("query_embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

### Asking a Question

In [ ]:
question = "Do high levels of procalcitonin in the early phase after pediatric liver transplantation indicate poor postoperative outcome?"

response = rag_pipeline.run(
    {
        "query_embedder": {"text": question},
        "prompt_builder": {"question": question},
        "answer_builder": {"query": question},
    }
)
print(response["answer_builder"]["answers"][0].data)

# Pipeline Evaluation with Haystack

The following metrics will be used to evaluate the pipeline:

- [Document Mean Reciprocal Rank](https://docs.haystack.deepset.ai/docs/documentmrrevaluator): Evaluates retrieved documents using ground truth labels. It checks at what rank ground truth documents appear in the list of retrieved documents.
- [Semantic Answer Similarity](https://docs.haystack.deepset.ai/docs/sasevaluator): Evaluates predicted answers using ground truth labels. It checks the semantic similarity of a predicted answer and the ground truth answer using a fine-tuned language model.
- [Faithfulness](https://docs.haystack.deepset.ai/docs/faithfulnessevaluator): Uses an LLM to evaluate whether a generated answer can be inferred from the provided contexts. Does not require ground truth labels.

Let's run our RAG pipeline with a set of 5 questions, and make sure we have the ground truth labels (both answers and documents) for these questions:

In [ ]:
import random

questions, ground_truth_answers, ground_truth_docs = zip(
    *random.sample(list(zip(all_questions, all_ground_truth_answers, all_documents)), 5)
)

Next, let's run our pipeline and make sure to track what our pipeline returns as answers, and which documents it retrieves:

In [ ]:
rag_answers = []
retrieved_docs = []

for question in list(questions):
    response = rag_pipeline.run(
        {
            "query_embedder": {"text": question},
            "prompt_builder": {"question": question},
            "answer_builder": {"query": question},
        }
    )
    print(f"Question: {question}")
    print("Answer from pipeline:")
    print(response["answer_builder"]["answers"][0].data)
    print("\n-----------------------------------\n")

    rag_answers.append(response["answer_builder"]["answers"][0].data)
    retrieved_docs.append(response["answer_builder"]["answers"][0].documents)

Now, construct an `eval_pipeline` that includes all evaluators for the metrics we want to evaluate our pipeline on.

In [ ]:
from haystack.components.evaluators.document_mrr import DocumentMRREvaluator
from haystack.components.evaluators.faithfulness import FaithfulnessEvaluator
from haystack.components.evaluators.sas_evaluator import SASEvaluator

eval_pipeline = Pipeline()
eval_pipeline.add_component("doc_mrr_evaluator", DocumentMRREvaluator())
eval_pipeline.add_component("faithfulness", FaithfulnessEvaluator(chat_generator=VLLMChatGenerator(model=JUDGE_MODEL_ID, 
                                                                                                   api_key=Secret.from_token(VLLM_API_KEY),
                                                                                                   generation_kwargs={"response_format": {"type": "json_object"}}, 
                                                                                                   api_base_url=API_URL)))
                                                                                                #    api_base_url="http://localhost:8000/v1")))
eval_pipeline.add_component("sas_evaluator", SASEvaluator(model="sentence-transformers/all-MiniLM-L6-v2"))

results = eval_pipeline.run(
    {
        "doc_mrr_evaluator": {
            "ground_truth_documents": list([d] for d in ground_truth_docs),
            "retrieved_documents": retrieved_docs,
        },
        "faithfulness": {
            "questions": list(questions),
            "contexts": list([d.content] for d in ground_truth_docs),
            "predicted_answers": rag_answers,
        },
        "sas_evaluator": {"predicted_answers": rag_answers, "ground_truth_answers": list(ground_truth_answers)},
    }
)

### Constructing an Evaluation Report

In [ ]:
from haystack.evaluation.eval_run_result import EvaluationRunResult

inputs = {
    "question": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "answer": list(ground_truth_answers),
    "predicted_answer": rag_answers,
}

evaluation_result = EvaluationRunResult(run_name="pubmed_rag_pipeline", inputs=inputs, results=results)
evaluation_result.aggregated_report()

In [ ]:
results_df = evaluation_result.detailed_report(output_format="df") # detailed report with the scores for each sample in the dataset
results_df

In [ ]:
import pandas as pd

top_3 = results_df.nlargest(3, "sas_evaluator")
bottom_3 = results_df.nsmallest(3, "sas_evaluator")
pd.concat([top_3, bottom_3])

# Pipeline Evaluation with DeepEval

In addition to evaluating the final responses of the LLM, it is important that we also evaluate the individual components of the RAG pipeline as they can significantly impact the overall performance. Therefore, there are different metrics to evaluate the retriever, the generator and the overall pipeline.

In [ ]:
from deepeval.models import LocalModel
from haystack_integrations.components.evaluators.deepeval import DeepEvalEvaluator, DeepEvalMetric

model = LocalModel(
    model=JUDGE_MODEL_ID, 
    api_key=VLLM_API_KEY,
    generation_kwargs={"response_format": {"type": "json_object"}},
    base_url=API_URL
)

### Contextul Precision

The contextual precision metric measures our RAG pipeline's retriever by evaluating whether items in our contexts that are relevant to the given input are ranked higher than irrelevant ones.

In [ ]:
context_precision_pipeline = Pipeline()
evaluator = DeepEvalEvaluator(metric=DeepEvalMetric.CONTEXTUAL_PRECISION, metric_params={"model":model})
context_precision_pipeline.add_component("evaluator", evaluator)

In [ ]:
inputs = {
    "questions": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "ground_truths": list(ground_truth_answers),
    "responses": rag_answers,
}

evaluation_results = context_precision_pipeline.run(
    {"evaluator": inputs}
)
print(evaluation_results["evaluator"]["results"])

### Contextual Recall

Contextual recall measures the extent to which the contexts aligns with the `ground truth`.

In [ ]:
context_recall_pipeline = Pipeline()
evaluator = DeepEvalEvaluator(metric=DeepEvalMetric.CONTEXTUAL_RECALL, metric_params={"model":model})
context_recall_pipeline.add_component("evaluator", evaluator)

In [ ]:
inputs = {
    "questions": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "ground_truths": list(ground_truth_answers),
    "responses": rag_answers,
}

evaluation_results = context_recall_pipeline.run(
    {"evaluator": inputs}
)
print(evaluation_results["evaluator"]["results"])

### Contextual Relevancy

The contextual relevancy metric measures the quality of our RAG pipeline's retriever by evaluating the overall relevance of the context for a given question.

In [ ]:
context_relevancy_pipeline = Pipeline()
evaluator = DeepEvalEvaluator(metric=DeepEvalMetric.CONTEXTUAL_RELEVANCE, metric_params={"model":model})
context_relevancy_pipeline.add_component("evaluator", evaluator)

In [ ]:
inputs = {
    "questions": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "responses": rag_answers,
}

evaluation_results = context_relevancy_pipeline.run(
    {"evaluator": inputs}
)
print(evaluation_results["evaluator"]["results"])

### Answer relevancy

The answer relevancy metric measures the quality of our RAG pipeline's response by evaluating how relevant the response is compared to the provided question.

In [ ]:
answer_relevancy_pipeline = Pipeline()
evaluator = DeepEvalEvaluator(metric=DeepEvalMetric.ANSWER_RELEVANCY, metric_params={"model":model})
answer_relevancy_pipeline.add_component("evaluator", evaluator)

In [ ]:
inputs = {
    "questions": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "responses": rag_answers,
}

evaluation_results = answer_relevancy_pipeline.run(
    {"evaluator": inputs}
)
print(evaluation_results["evaluator"]["results"])

### Faithfulness

The faithfulness metric measures the quality of our RAG pipeline's responses by evaluating whether the response factually aligns with the contents of context we provided.

In [ ]:
faithfulness_pipeline = Pipeline()
evaluator = DeepEvalEvaluator(metric=DeepEvalMetric.FAITHFULNESS, metric_params={"model":model} )
faithfulness_pipeline.add_component("evaluator", evaluator)

In [ ]:
inputs = {
    "questions": list(questions),
    "contexts": list([d.content] for d in ground_truth_docs),
    "responses": rag_answers,
}

evaluation_results = faithfulness_pipeline.run(
    {"evaluator": inputs}
)
print(evaluation_results["evaluator"]["results"])